In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import warnings

warnings.filterwarnings('ignore')

In [8]:
try:
    file_path = 'Embrapii_seleção_analista_2026_questao02_Relatório de Dados.xlsx'
    df_projetos = pd.read_excel(file_path, sheet_name='projetos')
    df_projetos_empresas = pd.read_excel(file_path, sheet_name='projetos_empresas')
    df_empresas = pd.read_excel(file_path, sheet_name='empresas')
    df_unidades = pd.read_excel(file_path, sheet_name='unidade_embrapii')
    print("Dados carregados com sucesso a partir do arquivo Excel.")
except Exception as e:
    print(f"Erro ao carregar o arquivo Excel: {e}")
    try:
        df_projetos = pd.read_csv('Embrapii_seleção_analista_2026_questao02_Relatório de Dados.xlsx - projetos.csv')
        df_projetos_empresas = pd.read_csv('Embrapii_seleção_analista_2026_questao02_Relatório de Dados.xlsx - projetos_empresas.csv')
        df_empresas = pd.read_csv('Embrapii_seleção_analista_2026_questao02_Relatório de Dados.xlsx - empresas.csv')
        df_unidades = pd.read_csv('Embrapii_seleção_analista_2026_questao02_Relatório de Dados.xlsx - unidade_embrapii.csv')
        print("Dados carregados com sucesso a partir dos arquivos CSV de fallback.")
    except Exception as ex:
        print(f"Erro crítico ao carregar os dados de qualquer fonte: {ex}")

Dados carregados com sucesso a partir do arquivo Excel.


In [9]:
try:
    df_projetos['valor_total'] = df_projetos['valor_embrapii'] + df_projetos['valor_empresa'] + df_projetos['valor_unidade_embrapii']
    
    df_merged = df_projetos.merge(df_unidades, on='unidade_embrapii', how='left')
    
    df_emp_proj = df_projetos_empresas.merge(df_empresas, on='empresa', how='left')
    df_emp_proj_agg = df_emp_proj.groupby('codigo_projeto')['porte'].agg(lambda x: ', '.join(x)).reset_index()
    
    df_final = df_merged.merge(df_emp_proj_agg, on='codigo_projeto', how='left')
    print("Estruturação e cruzamento dos dados realizados com sucesso.")
except Exception as e:
    print(f"Erro durante o processamento e cruzamento dos dados: {e}")

Estruturação e cruzamento dos dados realizados com sucesso.


In [10]:
try:
    total_projetos = df_projetos['codigo_projeto'].nunique()
    total_investimento = df_projetos['valor_total'].sum()
    total_embrapii = df_projetos['valor_embrapii'].sum()
    total_empresas_invest = df_projetos['valor_empresa'].sum()
    total_unidades_invest = df_projetos['valor_unidade_embrapii'].sum()
    
    status_counts = df_projetos['status'].value_counts()
    
    top_techs = df_projetos['tecnologia_habilitadora'].value_counts().head(6)
    
    porte_counts = df_emp_proj['porte'].value_counts()
    
    evolucao_projetos = df_projetos.copy()
    evolucao_projetos['ano_inicio'] = pd.to_datetime(evolucao_projetos['data_inicio']).dt.year
    projetos_por_ano = evolucao_projetos['ano_inicio'].value_counts().sort_index()
    print("Cálculo dos indicadores executivos concluído.")
except Exception as e:
    print(f"Erro no cálculo dos indicadores: {e}")

Cálculo dos indicadores executivos concluído.


In [11]:
try:
    pdf_path = 'Relatorio_Executivo_Embrapii.pdf'
    pdf = PdfPages(pdf_path)
    
    fig1 = plt.figure(figsize=(8.27, 11.69))
    fig1.suptitle('Relatório Executivo de Projetos Apoiados', fontsize=22, fontweight='bold', y=0.95, color='#003366')
    
    ax1 = fig1.add_axes([0.1, 0.8, 0.8, 0.1])
    ax1.axis('off')
    
    bbox_props = dict(boxstyle="round,pad=1", fc="#f8f9fa", ec="#cccccc", lw=2)
    ax1.text(0.15, 0.5, f"Projetos Apoiados\n\n{total_projetos:,}", fontsize=14, fontweight='bold', ha='center', va='center', bbox=bbox_props, color='#333333')
    ax1.text(0.5, 0.5, f"Investimento Total\n\nR$ {total_investimento/1e9:.2f} Bilhões", fontsize=14, fontweight='bold', ha='center', va='center', bbox=bbox_props, color='#333333')
    ax1.text(0.85, 0.5, f"Aporte Embrapii\n\nR$ {total_embrapii/1e9:.2f} Bilhões", fontsize=14, fontweight='bold', ha='center', va='center', bbox=bbox_props, color='#333333')
    
    ax2 = fig1.add_axes([0.1, 0.45, 0.35, 0.25])
    ax2.pie([total_embrapii, total_empresas_invest, total_unidades_invest], labels=['Embrapii', 'Empresas', 'Unidades'], autopct='%1.1f%%', startangle=140, colors=['#005b96', '#6497b1', '#b3cde0'], textprops={'fontsize': 10, 'weight': 'bold'})
    ax2.set_title('Distribuição de Investimentos', fontsize=14, fontweight='bold', color='#003366')
    
    ax3 = fig1.add_axes([0.55, 0.45, 0.35, 0.25])
    sns.barplot(y=status_counts.index, x=status_counts.values, ax=ax3, palette='Blues_r')
    ax3.set_title('Status Geral dos Projetos', fontsize=14, fontweight='bold', color='#003366')
    ax3.set_xlabel('Quantidade')
    ax3.set_ylabel('')
    
    ax4 = fig1.add_axes([0.1, 0.1, 0.8, 0.25])
    sns.lineplot(x=projetos_por_ano.index, y=projetos_por_ano.values, ax=ax4, marker='o', linewidth=3, markersize=8, color='#005b96')
    ax4.set_title('Evolução do Volume de Projetos Iniciados por Ano', fontsize=14, fontweight='bold', color='#003366')
    ax4.set_xlabel('Ano')
    ax4.set_ylabel('Quantidade de Projetos')
    ax4.grid(True, linestyle='--', alpha=0.5)
    
    pdf.savefig(fig1)
    plt.close(fig1)
    print("Frente do relatório (Página 1) gerada no PDF.")
except Exception as e:
    print(f"Erro ao desenhar ou salvar a Página 1: {e}")

Frente do relatório (Página 1) gerada no PDF.


In [12]:
try:
    fig2 = plt.figure(figsize=(8.27, 11.69))
    fig2.suptitle('Detalhamento Tecnológico e Perfil Empresarial', fontsize=22, fontweight='bold', y=0.95, color='#003366')
    
    ax5 = fig2.add_axes([0.1, 0.60, 0.8, 0.25])
    sns.barplot(y=top_techs.index, x=top_techs.values, ax=ax5, palette='viridis')
    ax5.set_title('Principais Tecnologias Habilitadoras (Top 6)', fontsize=14, fontweight='bold', color='#003366')
    ax5.set_xlabel('Quantidade de Projetos')
    ax5.set_ylabel('')
    
    ax6 = fig2.add_axes([0.25, 0.20, 0.5, 0.25])
    sns.barplot(x=porte_counts.index, y=porte_counts.values, ax=ax6, palette='Set2')
    ax6.set_title('Participação de Empresas por Porte', fontsize=14, fontweight='bold', color='#003366')
    ax6.set_ylabel('Volume de Empresas Envolvidas')
    ax6.set_xlabel('')
    ax6.tick_params(axis='x', rotation=45)
    
    for p in ax6.patches:
        ax6.annotate(format(p.get_height(), '.0f'), 
                   (p.get_x() + p.get_width() / 2., p.get_height()), 
                   ha = 'center', va = 'center', 
                   xytext = (0, 9), 
                   textcoords = 'offset points', fontweight='bold')

    pdf.savefig(fig2)
    plt.close(fig2)
    
    pdf.close()
    print(f"Verso do relatório (Página 2) gerada e PDF finalizado com sucesso. Arquivo salvo como: {pdf_path}")
except Exception as e:
    print(f"Erro ao desenhar ou finalizar a Página 2 no PDF: {e}")

Verso do relatório (Página 2) gerada e PDF finalizado com sucesso. Arquivo salvo como: Relatorio_Executivo_Embrapii.pdf
